# MEOK cross-runtime replay
Actual inference from the existing 980-parameter synthetic process model on Kaggle CPU.

This notebook verifies the published code and input bytes, evaluates 560 retained transitions, records all outputs and preserves the model's large worst-case errors. It does not train a model, use a GPU, call a paid provider or create a GSPC score. The optional Cloudflare check only observes endpoint reachability; absence of an application response is not counted as successful inference. No credentials are stored or requested.


In [ ]:
from pathlib import Path
import urllib.request, urllib.error, hashlib, json, importlib.util
ROOT=Path('/kaggle/working/meok-cross-runtime'); ROOT.mkdir(exist_ok=True)
PIN='c11f640ba2c47cdc89ca383062133e63666e5e43'
LOCKS={'replay.py': '25310a41e5db8701354adb02992c153536c634e746d4b5fce31a42c59e9dd01e', 'source-lock.json': '442d0563cad67a60eb9e78611ff9ead144d10610c2e165afcb0822d051eb70cc'}
class NoRedirect(urllib.request.HTTPRedirectHandler):
    def redirect_request(self,*args,**kwargs): return None
opener=urllib.request.build_opener(NoRedirect)
for name,expected in LOCKS.items():
    url='https://raw.githubusercontent.com/CSOAI-ORG/councilof-ai/'+PIN+'/harness/meok-cross-runtime/'+name
    with opener.open(url,timeout=30) as response: raw=response.read(500001)
    assert len(raw)<=500000 and hashlib.sha256(raw).hexdigest()==expected
    (ROOT/name).write_bytes(raw)
# Canonical GitHub mirrors avoid an unexpected hosted HF redirect while retaining original HF byte pins.
vendor=ROOT/'vendor';vendor.mkdir(exist_ok=True)
source_lock=json.loads((ROOT/'source-lock.json').read_text())
for name,record in source_lock['files'].items():
    assert '/' not in name and '\\' not in name
    url='https://raw.githubusercontent.com/CSOAI-ORG/councilof-ai/'+PIN+'/harness/meok-cross-runtime/vendor/'+name
    with opener.open(url,timeout=30) as response: raw=response.read(300001)
    assert len(raw)<=300000 and hashlib.sha256(raw).hexdigest()==record['sha256']
    (vendor/name).write_bytes(raw)
spec=importlib.util.spec_from_file_location('qualified_replay',ROOT/'replay.py')
module=importlib.util.module_from_spec(spec); spec.loader.exec_module(module)
report=module.run('Kaggle-CPU-pinned-replay',ROOT/'results')
assert report['rows']==560 and report['state']=='LOCAL_MODEL_REPLAY_PASSED'
# Optional visibility probe, NOT a substitution for observed hosted inference.
preview='https://0e29bc3e-csoai-process-replay.nicholastempleman.workers.dev/health'
observation={'url':preview,'application_execution_established':False,'inference_requests':0}
try:
    req=urllib.request.Request(preview,headers={'User-Agent':'CSOAI-cross-runtime-replay/1'})
    try: response=opener.open(req,timeout=20)
    except urllib.error.HTTPError as error: response=error
    with response:
        raw=response.read(20001);observation.update(http_status=response.status,raw_sha256=hashlib.sha256(raw).hexdigest(),cf_ray_present=bool(response.headers.get('cf-ray')),model_body_digest_present=bool(response.headers.get('x-content-sha256')))
        if len(raw)<=20000 and response.headers.get('x-content-sha256')==hashlib.sha256(raw).hexdigest():
            health=json.loads(raw);observation['application_health_established']=health.get('state')=='RESEARCH_PREVIEW'
        else: observation['bounded_error_excerpt']=raw.decode('utf-8','replace')[:160]
except Exception as exc: observation.update(error_type=type(exc).__name__)
(ROOT/'cloudflare-visibility.json').write_text(json.dumps(observation,indent=2)+'\n')
print('CLOUDFLARE_VISIBILITY_JSON='+json.dumps(observation))
(ROOT/'SOURCE.json').write_text(json.dumps({'canonical_repository':'CSOAI-ORG/councilof-ai','source_commit':PIN,'source_hashes':LOCKS,'model_weights_unchanged':True,'public_fixture_only':True,'cloudflare_health_is_not_inference':True},indent=2)+'\n')
print('KAGGLE_REPLAY_COMPLETED')


## Read the results
Download `meok-cross-runtime/results/report.json`, `predictions.jsonl` and `cloudflare-visibility.json`. Matching a reference result on another machine establishes portability for this declared test, not correctness in the real world or a new general capability. The model and dataset retain their original MIT notices. The published test bank is not a new blind evaluation.
